## Convert Classifications Sheets to Cleaned Raw Data

In [ ]:
import os, glob
import pandas as pd

In [ ]:

# ============================================================
# SETTINGS — edit these two paths
# ============================================================
# Folder that contains your offense CSVs (the inputs):
input_folder  = "end path to folder of CVSs here"
# Folder where cleaned versions are written. KEEP THIS SEPARATE from input_folder
output_folder = os.path.join(os.path.expanduser("~"), "Desktop", "cleaned_offenses")


# ============================================================
# RANK MAP  (Strict Liability lowest, Purposive highest)
rank_map = {
    "Strict Liability": 1,
    "Negligent": 2,
    "Reckless": 3,
    "Knowing": 4,
    "Purposive": 5,
}

# ============================================================
# CLEANING FUNCTIONS
# ============================================================
def clean_to_na_series(s: pd.Series) -> pd.Series:
    s = s.astype("string").str.strip()
    s = s.replace("", pd.NA)
    s = s.replace("No Mental States", pd.NA)
    return s

def min_rank_label(df: pd.DataFrame, cols, rank_map, na_label="n/a") -> pd.Series:
    """
    For each row, pick the single label among `cols` with the LOWEST rank
    (per `rank_map`) — i.e. the least culpable state present. If all are
    blank/"No Mental States"/NA, return `na_label`.
    (Change `.min` to `.max` below to keep the highest-ranked label instead.)
    """
    tmp = df[cols].copy()

    # clean each column
    for c in cols:
        tmp[c] = clean_to_na_series(tmp[c])

    # map labels to ranks
    ranks = tmp.apply(lambda col: col.map(rank_map))

    # get the lowest rank per row (skip NAs)  <-- change .min to .max to flip
    chosen_rank = ranks.min(axis=1, skipna=True)

    # map rank -> label (invert rank_map)
    inv_rank_map = {v: k for k, v in rank_map.items()}
    out = chosen_rank.map(inv_rank_map)

    # if all missing, fill with "n/a"
    return out.fillna(na_label).astype("string")


# ============================================================
# PER-FILE HELPERS
# ============================================================
# The element columns every file must contain (spelled exactly):
REQUIRED_ELEMENT_COLS = [
    "Action #1", "Action #2", "Action #3",
    "Result #1", "Result #2", "Result #3",
    "Circumstance #1", "Circumstance #2", "Circumstance #3",
]


def find_missing_columns(df: pd.DataFrame) -> list:
    """Return a list of required columns that are NOT in this file."""
    missing = []
    # a law-number column under either spelling is acceptable
    if "law number" not in df.columns and "law_number" not in df.columns:
        missing.append("law number (or law_number)")
    if "state" not in df.columns:
        missing.append("state")
    for c in REQUIRED_ELEMENT_COLS:
        if c not in df.columns:
            missing.append(c)
    return missing


def collect_unknown_labels(df: pd.DataFrame, known_labels: set) -> set:
    """Return label values that appear in the element columns but are NOT in rank_map."""
    found = set()
    for c in REQUIRED_ELEMENT_COLS:
        values = clean_to_na_series(df[c]).dropna().unique()
        for v in values:
            if v not in known_labels:
                found.add(v)
    return found


def clean_offense_df(df: pd.DataFrame) -> pd.DataFrame:
    """Collapse one offense file to a single label per element."""
    law_col = "law number" if "law number" in df.columns else "law_number"
    return pd.DataFrame({
        "law_number": df[law_col],
        "state": df["state"].astype("string").str.strip(),
        "action_merged":       min_rank_label(df, ["Action #1", "Action #2", "Action #3"], rank_map, na_label="n/a"),
        "result_merged":       min_rank_label(df, ["Result #1", "Result #2", "Result #3"], rank_map, na_label="n/a"),
        "circumstance_merged": min_rank_label(df, ["Circumstance #1", "Circumstance #2", "Circumstance #3"], rank_map, na_label="n/a"),
    })


# ============================================================
# RUN — process every CSV in the input folder
# ============================================================
os.makedirs(output_folder, exist_ok=True)

csv_paths = sorted(glob.glob(os.path.join(input_folder, "*.csv")))
print(f"Found {len(csv_paths)} CSV file(s) in {input_folder}\n")

known_labels = set(rank_map.keys())
unknown_report = {}   # label -> set of filenames it appeared in
written = 0
skipped = 0

for path in csv_paths:
    name = os.path.basename(path)

    # 1) try to read the file
    try:
        df = pd.read_csv(path)
    except Exception as e:
        print(f"SKIPPED {name} — could not read file: {e}")
        skipped += 1
        continue

    # 2) check it has the right columns

    missing = find_missing_columns(df)
    if missing:
        print(f"SKIPPED {name} — missing required column(s): {', '.join(missing)}")
        skipped += 1
        continue

    # 3) note any labels that aren't in rank_map (for the report at the end)
    for lbl in collect_unknown_labels(df, known_labels):
        unknown_report.setdefault(lbl, set()).add(name)

    # 4) clean it and write the result
    try:
        cleaned = clean_offense_df(df)
        offense_name = os.path.splitext(name)[0]          # "DUI.csv" -> "DUI"
        out_path = os.path.join(output_folder, f"{offense_name}_clean.csv")
        cleaned.to_csv(out_path, index=False)
        print(f"Wrote: {out_path}")
        written += 1
    except Exception as e:
        print(f"SKIPPED {name} — error while cleaning: {e}")
        skipped += 1

print(f"\nDone. {written} file(s) cleaned, {skipped} skipped.")

# ============================================================
# REPORT — labels found in the data that are NOT in rank_map
# (these were treated as missing and dropped from the results)
# ============================================================
if unknown_report:
    print("\n--- Labels found in the data that are NOT in rank_map ---")
    for lbl, files in sorted(unknown_report.items()):
        print(f"  {lbl!r}  (in: {', '.join(sorted(files))})")
    print("Fix the spelling in the data, or add these to rank_map if they are valid.")
else:
    print("\nAll labels in the data matched rank_map. No unexpected labels found.")